# 01 - EDA pentru toate bateriile NASA 5-56

Scopul acestui notebook este sa intelegem datasetul curatat inainte de modelare:

- cate cicluri utile are fiecare baterie;
- cum evolueaza capacitatea si SOH;
- ce baterii sunt potrivite pentru train/validation/test;
- ce semnale brute exista in fisierele de ciclu;
- ce tabele si figuri merita pastrate pentru lucrare.

Notebookul foloseste datasetul curatat din `data_test/` si salveaza tabele/figuri in `artifacts/`.

## 1. Imports, paths si configurare

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATASET_ROOT = REPO_ROOT / "data_test" / "Cleaned Datasets" / "Datasets 5-56 cleaned" / "cleaned_dataset"
METADATA_PATH = DATASET_ROOT / "metadata.csv"
CYCLE_DATA_DIR = DATASET_ROOT / "data"

ARTIFACTS_DIR = REPO_ROOT / "artifacts"
FIG_DIR = ARTIFACTS_DIR / "figures" / "eda"
TABLE_DIR = ARTIFACTS_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

assert METADATA_PATH.exists(), f"Missing metadata file: {METADATA_PATH}"
assert CYCLE_DATA_DIR.exists(), f"Missing cycle data directory: {CYCLE_DATA_DIR}"

print("Repository:", REPO_ROOT)
print("Dataset:", DATASET_ROOT)
print("Figures:", FIG_DIR)
print("Tables:", TABLE_DIR)

## 2. Functii utilitare

Datele vin din structuri MATLAB transformate in CSV, deci unele valori apar ca liste textuale. Functiile de mai jos parseaza robust timestamp-uri si valori numerice.

In [ ]:
def parse_start_time(value) -> pd.Timestamp:
    if pd.isna(value):
        return pd.NaT
    numbers = np.fromstring(str(value).replace("[", " ").replace("]", " ").replace(",", " "), sep=" ")
    if numbers.size < 6:
        return pd.NaT
    year, month, day, hour, minute, second = numbers[:6]
    sec_int = int(second)
    micro = int(round((float(second) - sec_int) * 1_000_000))
    try:
        return pd.Timestamp(
            year=int(year),
            month=int(month),
            day=int(day),
            hour=int(hour),
            minute=int(minute),
            second=sec_int,
            microsecond=micro,
        )
    except ValueError:
        return pd.NaT


def parse_numeric(value) -> float:
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    import re
    match = re.search(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?", str(value))
    if not match:
        return np.nan
    try:
        return float(match.group(0))
    except ValueError:
        return np.nan


def save_current_figure(name: str):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved: {path}")


def load_cycle_csv(filename: str) -> pd.DataFrame:
    path = CYCLE_DATA_DIR / str(filename)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

## 3. Incarcare metadata si audit initial

In [ ]:
metadata = pd.read_csv(METADATA_PATH)
metadata["start_time"] = metadata["start_time"].apply(parse_start_time)
metadata["Capacity"] = metadata["Capacity"].apply(parse_numeric)
metadata["Re"] = metadata["Re"].apply(parse_numeric)
metadata["Rct"] = metadata["Rct"].apply(parse_numeric)
metadata["filename"] = metadata["filename"].astype(str)
metadata = metadata.sort_values(["battery_id", "type", "start_time", "uid"], kind="mergesort")
metadata["type_cycle_index"] = metadata.groupby(["battery_id", "type"]).cumcount() + 1

mapped_files = set(metadata["filename"].dropna().astype(str))
existing_files = {p.name for p in CYCLE_DATA_DIR.glob("*.csv")}
missing_files = sorted(mapped_files - existing_files)

summary = {
    "metadata_rows": int(len(metadata)),
    "battery_count": int(metadata["battery_id"].nunique()),
    "cycle_file_count": int(len(existing_files)),
    "missing_mapped_files": int(len(missing_files)),
    "start_time_missing": int(metadata["start_time"].isna().sum()),
}
print(json.dumps(summary, indent=2))
print("\nCycle type counts:")
display(metadata["type"].value_counts().rename_axis("type").reset_index(name="rows"))
print("\nFirst rows:")
display(metadata.head())

## 4. Tabel de descarcari si tinte EDA

Pentru RUL folosim ciclurile de descarcare, deoarece `Capacity` este relevanta aici. `RUL` este definit initial ca numar de cicluri ramase pana la ultimul ciclu masurat pentru bateria respectiva.

In [ ]:
discharge = metadata.loc[metadata["type"] == "discharge"].copy()
discharge = discharge.dropna(subset=["battery_id", "start_time", "Capacity", "filename"])
discharge = discharge[discharge["Capacity"] > 0].copy()
discharge = discharge[discharge["filename"].isin(existing_files)].copy()
discharge = discharge.sort_values(["battery_id", "start_time", "uid"], kind="mergesort")
discharge["cycle_index"] = discharge.groupby("battery_id").cumcount() + 1

def add_eda_targets(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values("cycle_index").copy()
    g["battery_id"] = group.name
    raw_capacity = g["Capacity"].astype(float)
    positive = raw_capacity[raw_capacity > 0]
    median_capacity = float(positive.median()) if len(positive) else np.nan
    lower = 0.5 * median_capacity if np.isfinite(median_capacity) else 0.0
    upper = 1.4 * median_capacity if np.isfinite(median_capacity) else np.inf
    clean_capacity = raw_capacity.where((raw_capacity >= lower) & (raw_capacity <= upper), np.nan)
    clean_capacity = clean_capacity.interpolate(limit_direction="both").ffill().bfill()
    initial_capacity = float(clean_capacity.head(3).median())
    final_capacity = float(clean_capacity.tail(3).median())
    g["capacity_ah_clean"] = clean_capacity
    g["initial_capacity"] = initial_capacity
    g["soh"] = clean_capacity / initial_capacity if initial_capacity > 0 else np.nan
    g["rul_cycles"] = len(g) - g["cycle_index"]
    g["capacity_drop_from_initial"] = initial_capacity - clean_capacity
    g["final_capacity_median3"] = final_capacity
    return g

discharge = discharge.groupby("battery_id", group_keys=False).apply(add_eda_targets)

battery_summary = (
    discharge.groupby("battery_id")
    .agg(
        cycles=("cycle_index", "max"),
        first_date=("start_time", "min"),
        last_date=("start_time", "max"),
        initial_capacity=("initial_capacity", "first"),
        final_capacity=("final_capacity_median3", "first"),
        min_capacity=("capacity_ah_clean", "min"),
        max_capacity=("capacity_ah_clean", "max"),
        final_soh=("soh", "last"),
    )
    .reset_index()
)
battery_summary["capacity_drop_ah"] = battery_summary["initial_capacity"] - battery_summary["final_capacity"]
battery_summary["capacity_drop_pct"] = 100 * battery_summary["capacity_drop_ah"] / battery_summary["initial_capacity"]
battery_summary = battery_summary.sort_values(["cycles", "battery_id"], ascending=[False, True])

summary_path = TABLE_DIR / "battery_eda_summary.csv"
battery_summary.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)
display(battery_summary)

## 5. Cate cicluri utile are fiecare baterie?

Acest grafic ne ajuta sa alegem splituri echilibrate si sa observam bateriile cu istoric scurt.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_df = battery_summary.sort_values("cycles", ascending=False)
ax.bar(plot_df["battery_id"], plot_df["cycles"], color="#3a6ea5")
ax.axhline(20, color="red", linestyle="--", linewidth=1, label="20 cycles reference")
ax.set_title("Useful discharge cycles per battery")
ax.set_xlabel("Battery ID")
ax.set_ylabel("Discharge cycles")
ax.tick_params(axis="x", rotation=60)
ax.legend()
plt.tight_layout()
save_current_figure("01_cycles_per_battery.png")
plt.show()

## 6. Curbe de capacitate pentru toate bateriile

Graficul este aglomerat, dar util pentru imaginea generala. Pentru lucrare putem folosi si subsetul reprezentativ din celula urmatoare.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
for battery_id, g in discharge.groupby("battery_id"):
    ax.plot(g["cycle_index"], g["capacity_ah_clean"], linewidth=1.1, alpha=0.75, label=battery_id)
ax.set_title("Capacity degradation - all batteries")
ax.set_xlabel("Discharge cycle index")
ax.set_ylabel("Capacity (Ah)")
ax.legend(ncol=4, fontsize=8, frameon=True)
plt.tight_layout()
save_current_figure("02_capacity_all_batteries.png")
plt.show()

## 7. Baterii reprezentative pentru grafice clare

Selectam automat cateva baterii: clasicele NASA B0005/B0006/B0007/B0018 plus baterii cu istoric lung, mediu si scurt.

In [ ]:
classic = [b for b in ["B0005", "B0006", "B0007", "B0018"] if b in set(discharge["battery_id"])]
long_batteries = battery_summary.head(4)["battery_id"].tolist()
short_batteries = battery_summary.tail(4)["battery_id"].tolist()
median_cycle_count = battery_summary["cycles"].median()
mid_batteries = (
    battery_summary.assign(distance=lambda d: (d["cycles"] - median_cycle_count).abs())
    .sort_values(["distance", "battery_id"])
    .head(4)["battery_id"]
    .tolist()
)
REPRESENTATIVE_BATTERIES = list(dict.fromkeys(classic + long_batteries + mid_batteries + short_batteries))[:12]
print(REPRESENTATIVE_BATTERIES)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=False)
for battery_id in REPRESENTATIVE_BATTERIES:
    g = discharge[discharge["battery_id"] == battery_id]
    axes[0].plot(g["cycle_index"], g["capacity_ah_clean"], marker="o", markersize=2, linewidth=1.2, label=battery_id)
    axes[1].plot(g["cycle_index"], g["soh"], marker="o", markersize=2, linewidth=1.2, label=battery_id)
axes[0].set_title("Capacity - representative batteries")
axes[0].set_xlabel("Cycle index")
axes[0].set_ylabel("Capacity (Ah)")
axes[1].set_title("SOH - representative batteries")
axes[1].set_xlabel("Cycle index")
axes[1].set_ylabel("SOH = capacity / initial capacity")
axes[1].axhline(0.7, color="red", linestyle="--", linewidth=1, label="70% reference")
for ax in axes:
    ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
save_current_figure("03_representative_capacity_soh.png")
plt.show()

## 8. Degradare initial-final pe baterie

In [ ]:
plot_df = battery_summary.sort_values("capacity_drop_pct", ascending=False)
fig, ax = plt.subplots(figsize=(14, 6))
colors = np.where(plot_df["capacity_drop_pct"] >= 20, "#b84545", "#3a6ea5")
ax.bar(plot_df["battery_id"], plot_df["capacity_drop_pct"], color=colors)
ax.axhline(20, color="red", linestyle="--", linewidth=1, label="20% capacity fade reference")
ax.set_title("Observed capacity drop from first cycles to final cycles")
ax.set_xlabel("Battery ID")
ax.set_ylabel("Capacity drop (%)")
ax.tick_params(axis="x", rotation=60)
ax.legend()
plt.tight_layout()
save_current_figure("04_capacity_drop_pct.png")
plt.show()

## 9. Impedanta: valori plauzibile si outlieri

`Re` si `Rct` contin outlieri foarte mari sau valori imposibile. Pentru grafice folosim doar valori pozitive intr-un interval larg, iar in modelare le tratam robust.

In [ ]:
impedance = metadata.loc[metadata["type"] == "impedance"].copy()
impedance_clean = impedance[(impedance["Re"] > 0) & (impedance["Re"] <= 10) & (impedance["Rct"] > 0) & (impedance["Rct"] <= 10)].copy()
impedance_clean = impedance_clean.sort_values(["battery_id", "start_time"], kind="mergesort")
impedance_clean["impedance_index"] = impedance_clean.groupby("battery_id").cumcount() + 1

print("Raw impedance rows:", len(impedance))
print("Plausible impedance rows:", len(impedance_clean))
display(impedance_clean[["Re", "Rct"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(impedance_clean["Re"].dropna(), bins=40, color="#3a6ea5")
axes[0].set_title("Re distribution, plausible range")
axes[0].set_xlabel("Re")
axes[0].set_ylabel("Count")
axes[1].hist(impedance_clean["Rct"].dropna(), bins=40, color="#7a5c99")
axes[1].set_title("Rct distribution, plausible range")
axes[1].set_xlabel("Rct")
plt.tight_layout()
save_current_figure("05_impedance_distribution.png")
plt.show()

## 10. Exemple de semnale brute pe cicluri

Pentru cateva baterii reprezentative, incarcam primul, mijlociul si ultimul ciclu de descarcare. Asta arata ce informatii pot deveni feature-uri.

In [ ]:
def sample_discharge_cycles(discharge_df: pd.DataFrame, battery_id: str) -> pd.DataFrame:
    g = discharge_df[discharge_df["battery_id"] == battery_id].sort_values("cycle_index")
    if g.empty:
        return g
    positions = sorted(set([0, len(g) // 2, len(g) - 1]))
    return g.iloc[positions]

signal_batteries = REPRESENTATIVE_BATTERIES[:4]
for battery_id in signal_batteries:
    sample_rows = sample_discharge_cycles(discharge, battery_id)
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)
    for _, row in sample_rows.iterrows():
        cycle = load_cycle_csv(row["filename"])
        time = pd.to_numeric(cycle.get("Time"), errors="coerce")
        label = f"cycle {int(row['cycle_index'])}"
        for ax, col, ylabel in zip(
            axes,
            ["Voltage_measured", "Current_measured", "Temperature_measured"],
            ["Voltage (V)", "Current (A)", "Temperature (C)"],
        ):
            if col in cycle.columns:
                ax.plot(time, pd.to_numeric(cycle[col], errors="coerce"), linewidth=1.2, label=label)
                ax.set_ylabel(ylabel)
                ax.legend(fontsize=8)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Raw discharge signals - {battery_id}")
    plt.tight_layout()
    save_current_figure(f"06_raw_discharge_signals_{battery_id}.png")
    plt.show()

## 12. Modeling quality labels for the two benchmarks

Pentru modelare vom folosi doua scenarii:

- `all_eligible`: toate bateriile cu cel putin 20 cicluri utile;
- `clean_benchmark`: baterii cu cel putin 60 cicluri, capacitate initiala plauzibila si fara cresteri/SOH spikes majore.

Aceste criterii nu sterg datele. Ele separa un test strict de un benchmark principal mai coerent pentru RUL.

In [ ]:
quality = battery_summary.copy()
quality["soh_span"] = quality["max_capacity"] / quality["initial_capacity"] - quality["min_capacity"] / quality["initial_capacity"]
quality["clean_benchmark"] = (
    (quality["cycles"] >= 60)
    & (quality["initial_capacity"] >= 0.5)
    & ((quality["max_capacity"] / quality["initial_capacity"]) <= 1.20)
    & (quality["final_soh"] <= 1.05)
    & ((quality["min_capacity"] / quality["initial_capacity"]) <= 0.98)
)
quality["modeling_group"] = np.where(quality["clean_benchmark"], "clean_benchmark", "stress_test_or_excluded")
quality_path = TABLE_DIR / "battery_eda_modeling_groups.csv"
quality.to_csv(quality_path, index=False)
print("Saved modeling groups:", quality_path)
display(quality[["battery_id", "cycles", "initial_capacity", "final_soh", "capacity_drop_pct", "modeling_group"]].sort_values(["modeling_group", "battery_id"]))

fig, ax = plt.subplots(figsize=(12, 5))
plot_df = quality.sort_values(["modeling_group", "cycles"], ascending=[True, False])
colors = np.where(plot_df["clean_benchmark"], "#3a6ea5", "#b84545")
ax.bar(plot_df["battery_id"], plot_df["cycles"], color=colors)
ax.set_title("Batteries used in clean benchmark vs strict/stress pool")
ax.set_xlabel("Battery ID")
ax.set_ylabel("Discharge cycles")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
save_current_figure("07_modeling_groups.png")
plt.show()

## 11. Observatii pentru modelare

Concluzii practice pentru urmatoarele notebookuri:

- folosim descarcarile ca unitate principala de modelare;
- folosim toate bateriile utile, dar impartite pe `battery_id`;
- folosim datele de charge si impedance ca informatie contextuala aliniata temporal;
- salvam splitul si feature table-ul in notebookul 02, apoi il refolosim in notebookul 03;
- pastram raw `.mat` doar pentru provenienta si validare, nu ca pipeline principal.